# Madhusudan Ghee - exploratory data analysis

The owner reviews one number each month: total revenue. It has gone up every year. This notebook
checks whether that means the business is growing.

## What this notebook does

This is the exploration stage. The job is to understand the data, check it is sound, and work out
what the story is before writing anything formal.

Once the story is clear, the answers are written as MySQL in `sql/04_business_questions.sql`, and the
charts are built in Tableau. Three stages, three tools:

| Stage | Tool |
|---|---|
| Explore | Python (this notebook) |
| Answer | MySQL |
| Present | Tableau |

No charts are drawn here. All visuals for this project are built in Tableau, so there is one place
to look for them and one set of numbers behind them.

## 1. Load the data

In [ ]:
import pandas as pd
from pathlib import Path

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)

DATA = Path('../data')

sales     = pd.read_csv(DATA / 'sales.csv', parse_dates=['invoice_date'])
products  = pd.read_csv(DATA / 'products.csv')
customers = pd.read_csv(DATA / 'customers.csv')
salesmen  = pd.read_csv(DATA / 'salesmen.csv')

print('sales    ', sales.shape)
print('products ', products.shape)
print('customers', customers.shape)
print('salesmen ', salesmen.shape)
sales.head()

## 2. What does one row represent?

The first thing to establish about any table. It decides how everything gets counted from here on.

In [ ]:
print('rows              :', len(sales))
print('unique invoice_no :', sales.invoice_no.nunique())
print('rows per invoice  :', round(len(sales) / sales.invoice_no.nunique(), 2))

One row is a line on an invoice, not an invoice. An invoice with two pack sizes is two rows.

So counting invoices needs `nunique()` on `invoice_no`. Using `len()` would overstate the order count
by about 8%, with no error to warn you.

## 3. Is the data sound?

Two generic checks (missing values, duplicates) and two business rules. The business rules are the
useful ones: every carton is 15 litres, and every line is quantity times rate.

In [ ]:
print('missing values:')
print(sales.isnull().sum().to_string())
print()
print('duplicate rows :', sales.duplicated().sum())
print('date range     :', sales.invoice_date.min().date(), 'to', sales.invoice_date.max().date())
print()
print('litres == cartons x 15         :', (sales.litres == sales.cartons * 15).all())
print('sales_amount == cartons x rate :', (sales.sales_amount == sales.cartons * sales.rate_per_carton).all())

Nothing missing, no duplicates, three complete financial years, and both rules hold on every row.

Worth confirming rather than assuming. It means anything surprising later is a business surprise,
not a data problem.

## 4. Join the tables and add the financial year

`sales` holds codes. To group by city or salesman the dimension tables have to be attached.

The financial year runs April to March, so January 2025 belongs to FY2024-25. Using calendar years
would split each festive season across two years and give the owner numbers he does not recognise.

In [ ]:
df = (sales
      .merge(customers, on='account_code',  how='left')
      .merge(products,  on='sku',           how='left')
      .merge(salesmen,  on='salesman_code', how='left'))

print('rows after joins:', len(df), '(should still be 2742)')
print('failed lookups  :', df[['account_name', 'pack_size', 'salesman_name']].isnull().sum().sum())

In [ ]:
fy_start = df.invoice_date.dt.year.where(df.invoice_date.dt.month >= 4,
                                         df.invoice_date.dt.year - 1)
df['fy'] = 'FY' + fy_start.astype(str) + '-' + (fy_start + 1).astype(str).str[2:]
df['month_no'] = df.invoice_date.dt.month

df.groupby('fy').agg(first=('invoice_date', 'min'),
                     last=('invoice_date', 'max'),
                     rows=('fy', 'size'))

## 5. Revenue and volume together

Revenue can rise for two reasons: we sold more, or we charged more. Those point to opposite
decisions, so the two are never looked at separately.

Realised price is total revenue divided by total litres. Not the average of `rate_per_carton`, which
would weight a one-carton invoice the same as a fifty-carton one.

In [ ]:
yearly = df.groupby('fy').agg(
    invoices=('invoice_no', 'nunique'),
    accounts=('account_code', 'nunique'),
    litres=('litres', 'sum'),
    revenue=('sales_amount', 'sum'),
)
yearly['rate_per_litre'] = (yearly.revenue / yearly.litres).round(2)
yearly

In [ ]:
growth = (yearly[['litres', 'revenue', 'rate_per_litre']].pct_change() * 100).round(1)
growth.columns = ['volume %', 'revenue %', 'price %']
print('Year on year:')
print(growth.to_string())

first, last = yearly.iloc[0], yearly.iloc[-1]
print()
print('Across the three years:')
for label, col in [('revenue', 'revenue'), ('volume', 'litres'), ('price', 'rate_per_litre')]:
    print(f'  {label:<8} {100 * (last[col] / first[col] - 1):+5.1f}%')

Revenue grew 8.3%. Volume grew 2.9%. Price grew 5.3%.

Around two thirds of the revenue growth came from raising the rate. In the most recent year volume
fell 0.6% while revenue still rose 2.0%.

## 6. What are the customers doing?

If volume is flat, either there are fewer customers or each one buys less.

In [ ]:
counts = yearly[['accounts', 'litres']].copy()
counts['litres_per_account'] = (counts.litres / counts.accounts).round(1)
counts

The customer count went up, from 114 to 135. That is 18% more accounts for the same volume, so volume
per account fell from 152.4 to 132.4 litres, down 13.1%.

The business is signing more accounts each year just to stand still.

## 7. Is that decline real?

This is the step worth spending time on.

If the 21 accounts added were smaller than average, they would pull the average down even if no
existing customer had changed anything. The average would be falling for a harmless reason.

To settle it, hold the customer list still. Look only at accounts that bought in all three years and
watch what those accounts did. Any change there is real behaviour. This is a cohort.

In [ ]:
years_present = df.groupby('account_code')['fy'].nunique()
loyal = years_present[years_present == 3].index
print(f'Accounts buying in all three years: {len(loyal)} of {df.account_code.nunique()}')

cohort = (df[df.account_code.isin(loyal)]
          .groupby('fy')
          .agg(accounts=('account_code', 'nunique'), litres=('litres', 'sum')))
cohort['litres_per_account'] = (cohort.litres / cohort.accounts).round(1)
cohort

In [ ]:
print(f"Same 107 accounts, three years : {100 * (cohort.litres.iloc[-1] / cohort.litres.iloc[0] - 1):+.1f}%")
print(f"Final year alone               : {100 * (cohort.litres.iloc[-1] / cohort.litres.iloc[1] - 1):+.1f}%")

The same 107 accounts bought 8.9% less, and most of that happened in the final year.

So it is not a mix effect. Existing customers are buying less, and the gap is being covered by new
sign-ups and higher prices.

## 8. Is it a shift to smaller packs?

A move towards 200 ml packs would reduce litres per account without any loss of interest. Worth
ruling out.

In [ ]:
mix = df.pivot_table(index='fy', columns='pack_size', values='litres', aggfunc='sum')
(100 * mix.div(mix.sum(axis=1), axis=0)).round(1)

The mix barely moves. 1 L holds about 53% of litres in all three years. Not a pack-size effect.

## 9. Seasonality

The owner compares each month with the previous month and sees a collapse every December. Three
years are combined here so one unusual month cannot set the pattern.

In [ ]:
monthly = df.groupby('month_no')['litres'].sum()
monthly.index = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
share = (100 * monthly / monthly.sum()).round(1)
print(share.to_string())
print()
print(f'Peak   : Oct {share.max()}%')
print(f'Trough : Jul {share.min()}%')
print(f'Oct is {monthly.max() / monthly.min():.2f}x July')

October is 1.74 times July, and the October to December quarter carries 31.3% of the year.

The December drop is the slope off the October peak, not a problem. The month-on-month review is the
wrong comparison. This month against the same month last year would be the right one.

## 10. Salesmen

Total volume is unfair, since a salesman with more accounts should sell more. Litres per account is
the comparison that holds up. Realised rate is included to check nobody is buying volume with
discount.

In [ ]:
by_sm = df.groupby(['salesman_code', 'salesman_name']).agg(
    accounts=('account_code', 'nunique'),
    litres=('litres', 'sum'),
    revenue=('sales_amount', 'sum'),
)
by_sm['litres_per_account'] = (by_sm.litres / by_sm.accounts).round(0)
by_sm['rate_per_litre'] = (by_sm.revenue / by_sm.litres).round(2)
by_sm.sort_values('litres_per_account', ascending=False)

Deepak Rana gets 522 litres per account, 43% more than the next best, at the same realised rate as
everyone else.

Two explanations fit and the data cannot separate them: he sells better, or his territory (Kairana
and Deoband, the two strongest cities) is better. Asking him what he does differently costs nothing.

## 11. Export the Tableau extract

Tableau reads one flat table. The joins and the financial year are done here rather than inside the
workbook, so the logic sits in a file that can be read and version controlled.

In [ ]:
extract = df[['invoice_date', 'invoice_no', 'fy', 'month_no',
              'account_code', 'account_name', 'city', 'salesman_code', 'salesman_name',
              'sku', 'product_name', 'pack_size',
              'cartons', 'litres', 'rate_per_carton', 'sales_amount']].copy()

extract.to_csv('../tableau/ghee_sales_extract.csv', index=False)
print(f'Wrote {len(extract):,} rows to tableau/ghee_sales_extract.csv')
print(f'Revenue total: {extract.sales_amount.sum():,.0f}')
print(f'Litres total : {extract.litres.sum():,.0f}')

# Tableau cannot divide one total by another without an aggregate calculated
# field, and those are awkward to keep consistent across sheets. The two ratio
# charts read these small pre-aggregated files instead, where the division has
# already been done at the right level.
by_year = extract.groupby('fy').agg(
    litres=('litres', 'sum'), revenue=('sales_amount', 'sum'),
    accounts=('account_code', 'nunique'), invoices=('invoice_no', 'nunique')).reset_index()
by_year['rate_per_litre']     = (by_year.revenue / by_year.litres).round(2)
by_year['litres_per_account'] = (by_year.litres / by_year.accounts).round(1)
by_year.to_csv('../tableau/summary_by_year.csv', index=False)

by_salesman = extract.groupby('salesman_name').agg(
    litres=('litres', 'sum'), revenue=('sales_amount', 'sum'),
    accounts=('account_code', 'nunique')).reset_index()
by_salesman['litres_per_account'] = (by_salesman.litres / by_salesman.accounts).round(0)
by_salesman['rate_per_litre']     = (by_salesman.revenue / by_salesman.litres).round(2)
by_salesman = by_salesman.sort_values('litres_per_account', ascending=False)
by_salesman.to_csv('../tableau/summary_by_salesman.csv', index=False)

print('Wrote summary_by_year.csv and summary_by_salesman.csv')

## Summary

| # | Finding | Figure |
|---|---|---|
| 1 | Growth is mostly price | Revenue +8.3%, volume +2.9%, price +5.3% |
| 2 | Volume fell in the latest year | -0.6% while revenue rose 2.0% |
| 3 | More accounts, less from each | Accounts +18.4%, litres per account -13.1% |
| 4 | Established customers buying less | Same 107 accounts, -8.9% |
| 5 | Not a pack-size shift | 1 L held 53.7% to 53.2% of litres |
| 6 | Sharp seasonality | Oct 11.4% vs Jul 6.5% of annual litres |
| 7 | One salesman ahead | 522 L per account vs 291 to 365 |

Revenue is up because prices are up. Underlying demand is shrinking, hidden by new sign-ups and
annual rate increases. Price rises cannot continue indefinitely.

Next: `sql/04_business_questions.sql` for the same findings in MySQL, and
`tableau/TABLEAU_GUIDE.md` to build the dashboard.